In [3]:
import numpy as np
import pandas as pd
import torch
import joblib
import glob
import os

# Importujemy definicję sieci z Twojego głównego pliku!
from FirstDiagnosisSystemClass import GreyBoxSystem

# ====================================================================
# KONFIGURACJA ŚCIEŻEK I PARAMETRÓW (DOSTOSUJ DO SIEBIE!)
# ====================================================================
# Ścieżki do folderu z danymi (używamy glob, żeby złapać wszystkie pasujące pliki)
DATA_DIR = r"C:\Users\anton\OneDrive\Desktop\data"

HEALTHY_FILES = glob.glob(os.path.join(DATA_DIR, "*_NF*.csv"))
FAULTY_FILES_PIM = glob.glob(os.path.join(DATA_DIR, "*_f_pim*.csv"))
FAULTY_FILES_PIC = glob.glob(os.path.join(DATA_DIR, "*_f_pic*.csv"))
FAULTY_FILES_WAF = glob.glob(os.path.join(DATA_DIR, "*_f_waf*.csv"))

ALPHAS_TO_TEST = np.linspace(0.0001, 0.1, num=2000)
BURN_IN = 50   # Ignorujemy pierwsze 50 próbek (2.5 sekundy)
MARGIN = 1.4   # Margines bezpieczeństwa 40% ponad maksymalny szum

# ====================================================================
# FUNKCJE POMOCNICZE
# ====================================================================
def simulate_and_get_raw_error(model, scaler_u, scaler_y, input_cols, target_cols, file_path):
    df = pd.read_csv(file_path)
    
    # Wyciągamy fizyczne (prawdziwe) wartości
    u_raw = df[input_cols].values
    y_true_real = df[target_cols].values
    
    # Normalizujemy wejścia
    u_norm = torch.tensor(scaler_u.transform(u_raw), dtype=torch.float32)
    
    y_preds_norm = []
    x_t = torch.zeros(1, model.num_states) # Stan początkowy
    
    model.eval()
    with torch.no_grad():
        # Symulacja KROK PO KROKU używając .step() (Dokładnie jak w FirstDiagnosisSystemClass)
        for t in range(len(u_norm)):
            u_t = u_norm[t:t+1] # Pobieramy jedną próbkę (kształt 1 x num_inputs)
            y_hat_t, x_t = model.step(u_t, x_t)
            y_preds_norm.append(y_hat_t.numpy()[0])
            
    # Odwracamy normalizację wyników sieci
    y_pred_real = scaler_y.inverse_transform(y_preds_norm)
    
    # Zwracamy surowy błąd (różnicę w Paskalach)
    return np.abs(y_true_real - y_pred_real).flatten()
def apply_ema_filter(e_raw, alpha):
    e_filtered = np.zeros_like(e_raw)
    e_filtered[0] = e_raw[0]
    for i in range(1, len(e_raw)):
        e_filtered[i] = alpha * e_raw[i] + (1 - alpha) * e_filtered[i-1]
    return e_filtered

def optimize_model(model_name, model, scaler_u, scaler_y, input_cols, target_cols, healthy_files, faulty_files):
    print(f"\n{'='*50}")
    print(f"ROZPOCZYNAM KALIBRACJĘ DLA: {model_name}")
    print(f"{'='*50}")
    
    if not healthy_files or not faulty_files:
        print("[!] Brak plików z danymi w podanych ścieżkach!")
        return
    
    # Krok 1: Zbieranie surowych błędów, by nie przetwarzać sieci wielokrotnie
    print("Symulacja na plikach ZDROWYCH...")
    healthy_errors_raw = [simulate_and_get_raw_error(model, scaler_u, scaler_y, input_cols, target_cols, f) for f in healthy_files]
    
    print("Symulacja na plikach Z USTERKĄ...")
    faulty_errors_raw = [simulate_and_get_raw_error(model, scaler_u, scaler_y, input_cols, target_cols, f) for f in faulty_files]

    best_alpha = None
    best_snr = 0.0
    best_threshold = 0.0
    
    print(f"\n{'-'*65}")
    print(f"{'Alpha':<8} | {'Max Szum (NF)':<15} | {'Max Usterka':<15} | {'Zalecany Próg':<15} | {'SNR'}")
    print(f"{'-'*65}")
    
    # Krok 2: Grid Search dla Alpha
    for alpha in ALPHAS_TO_TEST:
        max_noise = 0.0
        # Szukamy absolutnie największego szumu w zdrowych plikach
        for e_raw in healthy_errors_raw:
            e_filt = apply_ema_filter(e_raw, alpha)
            current_max = np.max(e_filt[BURN_IN:])
            if current_max > max_noise: max_noise = current_max
                
        max_fault = 0.0
        # Szukamy błędu w plikach z usterką
        for e_raw in faulty_errors_raw:
            e_filt = apply_ema_filter(e_raw, alpha)
            current_max = np.max(e_filt[BURN_IN:])
            if current_max > max_fault: max_fault = current_max
                
        threshold = max_noise * MARGIN
        snr = max_fault / max_noise if max_noise > 0 else 0
        
        # Wyświetlanie wyników w tabeli
        print(f"{alpha:<8} | {max_noise:<15.1f} | {max_fault:<15.1f} | {threshold:<15.1f} | {snr:.2f}")
        
        # Zapamiętujemy parametry z najwyższym SNR
        if snr > best_snr:
            best_snr = snr
            best_alpha = alpha
            best_threshold = threshold
            
    print(f"{'-'*65}")
    print(f"🏆 WYNIK: Najlepsze Alpha: {best_alpha} | Ostateczny Próg: {best_threshold:.1f} | SNR: {best_snr:.2f}")

# ====================================================================
# GŁÓWNA PĘTLA WYKONAWCZA
# ====================================================================
if __name__ == "__main__":
    print("Ładowanie modeli i skalerów...")
    
    # --- MSO 0 ---
    u0_cols = ['Intercooler_pressure', 'intercooler_temperature', 'throttle_position', 'engine_speed']
    y0_cols = ['intake_manifold_pressure']
    model0 = GreyBoxSystem(num_states=1, num_inputs=4)
    model0.load_state_dict(torch.load('szara_skrzynka_mso0_wagi.pth'))
    scaler_u0 = joblib.load('scaler_u_mso0.pkl')
    scaler_y0 = joblib.load('scaler_y_mso0.pkl')
    
    # --- MSO 10 ---
    u10_cols = ['air_mass_flow', 'intercooler_temperature', 'intake_manifold_pressure', 'throttle_position']
    y10_cols = ['Intercooler_pressure']
    model10 = GreyBoxSystem(num_states=1, num_inputs=4)
    model10.load_state_dict(torch.load('szara_skrzynka_mso10_wagi.pth'))
    scaler_u10 = joblib.load('scaler_u_mso10.pkl')
    scaler_y10 = joblib.load('scaler_y_mso10.pkl')
    
    # --- MSO 1 ---
    u1_cols = ['ambient_pressure', 'ambient_temperature', 'intercooler_temperature', 'throttle_position', 'engine_speed', 'injected_fuel_mass', 'wastegate_position']
    y1_cols = ['intake_manifold_pressure']
    model1 = GreyBoxSystem(num_states=5, num_inputs=7)
    model1.load_state_dict(torch.load('szara_skrzynka_mso1_wagi.pth'))
    scaler_u1 = joblib.load('scaler_u_mso1.pkl')
    scaler_y1 = joblib.load('scaler_y_mso1.pkl')

    # Uruchomienie optymalizacji
    # Dla MSO 0 badamy f_pim
    optimize_model("MSO 0 (Kolektor)", model0, scaler_u0, scaler_y0, u0_cols, y0_cols, HEALTHY_FILES, FAULTY_FILES_PIM)
    
    # Dla MSO 10 badamy f_pic (Możesz zmienić na FAULTY_FILES_WAF, jeśli chcesz dostroić pod przepływkę)
    optimize_model("MSO 10 (Intercooler)", model10, scaler_u10, scaler_y10, u10_cols, y10_cols, HEALTHY_FILES, FAULTY_FILES_PIC)
    
    # Dla MSO 1 badamy f_pim
    optimize_model("MSO 1 (Pełny Silnik)", model1, scaler_u1, scaler_y1, u1_cols, y1_cols, HEALTHY_FILES, FAULTY_FILES_PIM)
    
    print("\nKalibracja zakończona! Przepisz parametry 🏆 WYNIK do FirstDiagnosisSystemClass.py.")

Ładowanie modeli i skalerów...

ROZPOCZYNAM KALIBRACJĘ DLA: MSO 0 (Kolektor)
Symulacja na plikach ZDROWYCH...
Symulacja na plikach Z USTERKĄ...

-----------------------------------------------------------------
Alpha    | Max Szum (NF)   | Max Usterka     | Zalecany Próg   | SNR
-----------------------------------------------------------------
0.0001   | 12480.6         | 8541.1          | 17472.8         | 0.68
0.0001499749874937469 | 12454.9         | 9309.0          | 17436.8         | 0.75
0.00019994997498749375 | 12429.2         | 9797.8          | 17400.9         | 0.79
0.00024992496248124065 | 12403.7         | 10159.9         | 17365.1         | 0.82
0.0002998999499749875 | 12378.1         | 10450.0         | 17329.4         | 0.84
0.0003498749374687344 | 12352.7         | 10692.1         | 17293.7         | 0.87
0.00039984992496248126 | 12327.3         | 10895.9         | 17258.2         | 0.88
0.0004498249124562281 | 12301.9         | 11068.3         | 17222.7         | 0.90
